In [ ]:
# !pip install nltk sentence-transformers torch huggingface_hub tqdm skimpy wordcloud mlflow torch datasets netron skorch xgboost lightgbm imblearn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import skimpy as sk
import nltk
import mlflow
import mlflow.sklearn
import torch
import datasets
import string
from tqdm import tqdm
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')


#First Baseline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import *
from sklearn.metrics import *
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import ElasticNet
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from transformers import AutoTokenizer, AutoModelForSequenceClassification #Representation Learning, 2nd Baseline
from datasets import load_dataset

#3rd Final Baseline
import torch.nn as nn
import torch
from torch.optim import *
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [ ]:
test = pd.read_csv("/content/drive/MyDrive/CaresAI/tnm_staging/data_v2/raw/Task6_test_hard.csv")
train = pd.read_csv("/content/drive/MyDrive/CaresAI/tnm_staging/data_v2/raw/tcga_tnm_train.csv")
val = pd.read_csv("/content/drive/MyDrive/CaresAI/tnm_staging/data_v2/raw/tcga_tnm_val_no_res.csv")


In [ ]:
train.head()

,patient_filename,text,t,n,m
0,TCGA-05-4244.3a844132-f813-4d8e-8f7d-dbae0b23d7fd,Diagnosis: 1. Resection material from the righ...,2.0,2.0,1.0
1,TCGA-05-4245.902fe548-5b93-49c9-81db-2af4a4a88f3c,Diagnosis: 1. Atypical resection material from...,2.0,2.0,NaN
2,TCGA-05-4249.7e920317-d5c2-4160-9e2b-ef0101eb5a23,Diagnosis: 2. and 3. Two-part resection materi...,2.0,0.0,0.0
3,TCGA-05-4250.5574f2f8-f247-40e6-a285-7793edcf5358,2. and 3. Resection material from the right lo...,3.0,1.0,0.0
4,TCGA-05-4382.952c0f32-1472-49e1-8334-b0f1de4ac921,Main diagnosis/diagnoses: Primarily evidently ...,NaN,0.0,0.0


In [ ]:
train_t = train[["patient_filename", "text", "t"]]
train_t.dropna(inplace=True)
train_n = train[["patient_filename", "text", "n"]]
train_n.dropna(inplace=True)
train_m = train[["patient_filename", "text", "m"]]
train_m.dropna(inplace=True)

/tmp/ipykernel_1357/1489307519.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_t.dropna(inplace=True)
/tmp/ipykernel_1357/1489307519.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_n.dropna(inplace=True)
/tmp/ipykernel_1357/1489307519.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_m.dropna(inplace=True)


In [ ]:
en_t = train_t.copy()
en_n = train_n.copy()
en_m = train_m.copy()

In [ ]:
#en_t = en_t[["text", "ajcc_pathologic_t", "target"]]
#en_n = en_n[["text",	"ajcc_pathologic_n", "target"]]
#en_m = en_m[["text",	"ajcc_pathologic_m", "target"]]

In [ ]:
stop_words = set(stopwords.words('english'))

# keep negations because they matter clinically
negations = {"no", "not", "without"}
stop_words = stop_words - negations

lemmatizer = WordNetLemmatizer()

def clean_text(text):

    # Lowercase
    text = text.lower()

    # Remove URLs / weird PDF artifacts
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove special characters but keep numbers and medical units
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenize
    tokens = nltk.word_tokenize(text)

    # Remove stopwords + short tokens
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]

    # Lemmatization
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return " ".join(tokens)


def clean_df(df):

    df = df.copy()

    # Clean pathology report text
    df["clean_text"] = df["text"].apply(clean_text)

    return df

In [ ]:
# Apply preprocessing
en_t = clean_df(en_t).drop(columns=["text"], axis=1)
en_n = clean_df(en_n).drop(columns=["text"], axis=1)
en_m = clean_df(en_m).drop(columns=["text"], axis=1)

In [ ]:
en_t = en_t.drop(columns=["patient_filename"], axis=1)
en_n = en_n.drop(columns=["patient_filename"], axis=1)
en_m = en_m.drop(columns=["patient_filename"], axis=1)

In [ ]:
display(en_t.head(3), en_n.head(3), en_m.head(3))

,t,clean_text
0,2.0,diagnosis resection material right lower lobe ...
1,2.0,diagnosis atypical resection material indicate...
2,2.0,diagnosis two part resection material right lo...


,n,clean_text
0,2.0,diagnosis resection material right lower lobe ...
1,2.0,diagnosis atypical resection material indicate...
2,0.0,diagnosis two part resection material right lo...


,m,clean_text
0,1.0,diagnosis resection material right lower lobe ...
2,0.0,diagnosis two part resection material right lo...
3,0.0,resection material right lower lobe locally ad...


In [ ]:
en_t = en_t.rename(columns={"t": "target"})
en_t["targetn"] = en_t["target"] - 1
en_n = en_n.rename(columns={"n": "target"})
en_m = en_m.rename(columns={"m": "target"})


In [ ]:
display(en_t.head(), en_t["targetn"].unique(), en_t["target"].unique())

,target,clean_text,targetn
0,2.0,diagnosis resection material right lower lobe ...,1.0
1,2.0,diagnosis atypical resection material indicate...,1.0
2,2.0,diagnosis two part resection material right lo...,1.0
3,3.0,resection material right lower lobe locally ad...,2.0
5,2.0,lung resection material peripheral manifestati...,1.0


array([1., 2., 0., 3.])

array([2., 3., 1., 4.])

In [ ]:
en_t = en_t.drop(columns=["target"], axis=1)
en_t = en_t.rename(columns={"targetn": "target"})

##Training and Evaluation

In [ ]:
#Models (T)
t_lr = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(3, 5),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        max_features=150000
    )),
    ('clf', LogisticRegression(
        C=4.0,
        penalty='l2',
        solver='liblinear',   # use 'saga' if dataset is large
        class_weight='balanced',
        max_iter=2000
    ))
])


t_rf = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.9,
        sublinear_tf=True,
        max_features=50000     # ↓ important for trees
    )),
    ('clf', RandomForestClassifier(
        n_estimators=300,
        max_depth=30,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        class_weight='balanced_subsample',
        n_jobs=-1
    ))
])

t_lgb = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.9,
        sublinear_tf=True,
        max_features=50000
    )),
    ('clf', LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=64,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        class_weight='balanced',
        n_jobs=-1
    ))
])

## Separate Models

In [ ]:
#Models
n_lr = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(3, 5),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        max_features=150000
    )),
    ('clf', LogisticRegression(
        C=4.0,
        penalty='l2',
        solver='liblinear',   # use 'saga' if dataset is large
        class_weight='balanced',
        max_iter=2000
    ))
])



n_rf = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.9,
        sublinear_tf=True,
        max_features=50000     # ↓ important for trees
    )),
    ('clf', RandomForestClassifier(
        n_estimators=300,
        max_depth=30,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        class_weight='balanced_subsample',
        n_jobs=-1
    ))
])

n_lgb = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.9,
        sublinear_tf=True,
        max_features=50000
    )),
    ('clf', LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=64,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        class_weight='balanced',
        n_jobs=-1
    ))
])

In [ ]:
#Models (m)
m_lr = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(3, 5),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        max_features=150000
    )),
    ('clf', LogisticRegression(
        C=4.0,
        penalty='l2',
        solver='liblinear',   # use 'saga' if dataset is large
        class_weight='balanced',
        max_iter=2000
    ))
])


m_rf = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.9,
        sublinear_tf=True,
        max_features=50000     # ↓ important for trees
    )),
    ('clf', RandomForestClassifier(
        n_estimators=300,
        max_depth=30,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        class_weight='balanced_subsample',
        n_jobs=-1
    ))
])

m_lgb = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.9,
        sublinear_tf=True,
        max_features=50000
    )),
    ('clf', LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=64,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        class_weight='balanced',
        n_jobs=-1
    ))
])

In [ ]:
X = en_t["clean_text"]
y = en_t["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

t_models = {
    "lr": t_lr,
    "rf": t_rf,
    "lgb": t_lgb
}

n_models = {
    "lr": n_lr,
    "rf": n_rf,
    "lgb": n_lgb
}
m_models = {
    "lr": m_lr,
    "rf": m_rf,
    "lgb": m_lgb
}

In [ ]:
mlflow.set_experiment("TNM_T_Classification")


for name, model in t_models.items():

    with mlflow.start_run(run_name=name):

        # Train
        model.fit(X_train, y_train)

        # Predict
        y_pred = model.predict(X_test)

        # Probabilities for AUC
        try:
            y_proba = model.predict_proba(X_test)
            auc = roc_auc_score(y_test, y_proba, multi_class='ovr')
        except:
            auc = np.nan

        # Metrics
        f1 = f1_score(y_test, y_pred, average='macro')


        mlflow.log_metric("f1_macro", f1)
        mlflow.log_metric("auc_ovr", auc if not np.isnan(auc) else 0)

        mlflow.log_param("model", name)
        mlflow.log_param("tfidf_type", "char" if name == "logreg" else "word")

        # Class distribution (nice for paper)
        mlflow.log_dict(
            y_train.value_counts().to_dict(),
            "class_distribution.json"
        )


        mlflow.sklearn.log_model(model, name)

        print(f"{name} → F1: {f1:.4f}, AUC: {auc:.4f}")

2026/04/27 13:32:22 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/27 13:32:22 INFO mlflow.store.db.utils: Updating database tables
2026/04/27 13:32:23 INFO mlflow.tracking.fluent: Experiment with name 'TNM_T_Classification' does not exist. Creating a new experiment.
2026/04/27 13:32:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/27 13:32:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


lr → F1: 0.7170, AUC: 0.9131


2026/04/27 13:33:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/27 13:33:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


rf → F1: 0.6546, AUC: 0.8859
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.942166 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 374591
[LightGBM] [Info] Number of data points in the train set: 4682, number of used features: 11863
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/04/27 13:39:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/27 13:39:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


lgb → F1: 0.7514, AUC: 0.9348


In [ ]:
X = en_n["clean_text"]
y = en_n["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

t_models = {
    "lr": t_lr,
    "rf": t_rf,
    "lgb": t_lgb
}

n_models = {
    "lr": n_lr,
    "rf": n_rf,
    "lgb": n_lgb
}

m_models = {
    "lr": m_lr,
    "rf": m_rf,
    "lgb": m_lgb
}

In [ ]:
mlflow.set_experiment("TNM_N_Classification")


for name, model in n_models.items():

    with mlflow.start_run(run_name=name):

        # Train
        model.fit(X_train, y_train)

        # Predict
        y_pred = model.predict(X_test)

        # Probabilities for AUC
        try:
            y_proba = model.predict_proba(X_test)
            auc = roc_auc_score(y_test, y_proba, multi_class='ovr')
        except:
            auc = np.nan

        # Metrics
        f1 = f1_score(y_test, y_pred, average='macro')


        mlflow.log_metric("f1_macro", f1)
        mlflow.log_metric("auc_ovr", auc if not np.isnan(auc) else 0)

        mlflow.log_param("model", name)
        mlflow.log_param("tfidf_type", "char" if name == "logreg" else "word")

        # Class distribution (nice for paper)
        mlflow.log_dict(
            y_train.value_counts().to_dict(),
            "class_distribution.json"
        )


        mlflow.sklearn.log_model(model, name)

        print(f"{name} → F1: {f1:.4f}, AUC: {auc:.4f}")

2026/04/27 13:39:21 INFO mlflow.tracking.fluent: Experiment with name 'TNM_N_Classification' does not exist. Creating a new experiment.
2026/04/27 13:39:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/27 13:39:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


lr → F1: 0.6666, AUC: 0.9211


2026/04/27 13:40:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/27 13:40:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


rf → F1: 0.5942, AUC: 0.8795
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.590948 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 308355
[LightGBM] [Info] Number of data points in the train set: 3860, number of used features: 9984
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [W

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/04/27 13:43:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/27 13:43:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


lgb → F1: 0.7520, AUC: 0.9540


In [ ]:
X = en_m["clean_text"]
y = en_m["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


t_models = {
    "lr": t_lr,
    "rf": t_rf,
    "lgb": t_lgb
}

n_models = {
    "lr": n_lr,
    "rf": n_rf,
    "lgb": n_lgb
}

m_models = {
    "lr": m_lr,
    "rf": m_rf,
    "lgb": m_lgb
}

In [ ]:
mlflow.set_experiment("TNM_M_Classification")


for name, model in m_models.items():

    with mlflow.start_run(run_name=name):

        # Train
        model.fit(X_train, y_train)

        # Predict
        y_pred = model.predict(X_test)

        # Probabilities for AUC
        try:
            y_proba = model.predict_proba(X_test)[:, 1]
            auc = roc_auc_score(y_test, y_proba, multi_class='ovr')
        except:
            auc = np.nan

        # Metrics
        f1 = f1_score(y_test, y_pred, average='macro')


        mlflow.log_metric("f1_macro", f1)
        mlflow.log_metric("auc_ovr", auc if not np.isnan(auc) else 0)

        mlflow.log_param("model", name)
        mlflow.log_param("tfidf_type", "char" if name == "logreg" else "word")

        # Class distribution (nice for paper)
        mlflow.log_dict(
            y_train.value_counts().to_dict(),
            "class_distribution.json"
        )


        mlflow.sklearn.log_model(model, name)

        print(f"{name} → F1: {f1:.4f}, AUC: {auc:.4f}")

2026/04/27 13:43:57 INFO mlflow.tracking.fluent: Experiment with name 'TNM_M_Classification' does not exist. Creating a new experiment.
2026/04/27 13:44:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/27 13:44:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


lr → F1: 0.6593, AUC: 0.8438


2026/04/27 13:44:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/27 13:44:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


rf → F1: 0.5353, AUC: 0.7497
[LightGBM] [Info] Number of positive: 213, number of negative: 2919
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.343150 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 217992
[LightGBM] [Info] Number of data points in the train set: 3132, number of used features: 7271
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furth

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/04/27 13:45:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/27 13:45:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


lgb → F1: 0.6790, AUC: 0.8413


## Validation Set

In [ ]:
val_data = val.copy()

val_data_clean = val_data.apply(lambda x: clean_text(x["text"]), axis=1)

models = {
    "t_lgb": t_lgb,
    "n_lgb": n_lgb,
    "m_lgb": m_lgb
}

t_preds = models["t_lgb"].predict(val_data_clean)
n_preds = models["n_lgb"].predict(val_data_clean)
m_preds = models["m_lgb"].predict(val_data_clean)

submission = pd.DataFrame({
  "patient_filename": val_data["patient_filename"],
  "text": val_data["text"],
  "t": t_preds,
  "n": n_preds,
  "m": m_preds
})

submission.to_csv("pred.csv", index=False)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
test.head()

,id,text
0,TNM_0000,Female. Clinical diagnosis: Invasive ductal ca...
1,TNM_0001,Gross Description: Received in formalin is a r...
2,TNM_0002,Diagnosis / findings. Segmental resection of s...
3,TNM_0003,Gross Description: Received is a partial colec...
4,TNM_0004,Clinical history: 68-year-old man with intermi...


Test Set

In [ ]:
test_data = test.copy()

test_data_clean = test_data.apply(lambda x: clean_text(x["text"]), axis=1)

models = {
    "t_lgb": t_lgb,
    "n_lgb": n_lgb,
    "m_lgb": m_lgb
}

t_preds = models["t_lgb"].predict(test_data_clean)
n_preds = models["n_lgb"].predict(test_data_clean)
m_preds = models["m_lgb"].predict(test_data_clean)

test_submission = pd.DataFrame({
  "id": test_data["id"],
  "text": test_data["text"],
  "t": t_preds,
  "n": n_preds,
  "m": m_preds
})

test_submission.to_csv("test_pred.csv", index=False)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Statistical Test